# 🏫 Instituto Infnet - Pós-Graduação em Inteligência Artificial
## Redes Neurais Profundas (Deep Learning e Visão Computacional)
### Aula 5 - Parte 2: Classificação do MNIST com CNN (Convolução & Pooling)

**Instrutor:** Prof. Allan Spadini  
**Objetivo da Parte 2:** Resolver o mesmo problema do MNIST utilizando uma **Rede Neural Convolucional (CNN)** com `nn.Conv2d` e `nn.MaxPool2d`. Comparar a eficiência, contagem de parâmetros e acurácia em relação à MLP da Parte 1.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)


---
### 1. Importação de Bibliotecas & Dispositivo


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo em uso: {device}")


---
### 2. Carregamento dos Dados MNIST


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

batch_size = 64
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)


---
### 3. Arquitetura da Rede Neural Convolucional (`MNISTCNN`)
A arquitetura convolucional preserva a grade $2D$ da imagem $(1, 28, 28)$ e aplica:
1. **Bloco Convolucional 1:** `Conv2d(1, 16, kernel_size=3, padding=1)` + `ReLU` + `MaxPool2d(2, 2)` $\rightarrow$ Saída $(16, 14, 14)$
2. **Bloco Convolucional 2:** `Conv2d(16, 32, kernel_size=3, padding=1)` + `ReLU` + `MaxPool2d(2, 2)` $\rightarrow$ Saída $(32, 7, 7)$
3. **Classificador Denso:** `Flatten()` + `Linear(32 * 7 * 7, 128)` + `ReLU` + `Linear(128, 10)`


In [ ]:
class MNISTCNN(nn.Module):
    def __init__(self):
        super(MNISTCNN, self).__init__()
        
        # Bloco Convolucional 1: (1, 28, 28) -> (16, 14, 14)
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        # Bloco Convolucional 2: (16, 14, 14) -> (32, 7, 7)
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        # Classificador Linear Final
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
        
    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        logits = self.classifier(x)
        return logits

model_cnn = MNISTCNN().to(device)
print(model_cnn)

total_params_cnn = sum(p.numel() for p in model_cnn.parameters() if p.requires_grad)
print(f"Total de parâmetros treináveis da CNN: {total_params_cnn:,}")


---
### 4. Treinamento da CNN


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_cnn.parameters(), lr=0.001)

epochs = 5
train_losses_cnn = []
train_accs_cnn = []

for epoch in range(epochs):
    model_cnn.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs = model_cnn(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100
    
    train_losses_cnn.append(epoch_loss)
    train_accs_cnn.append(epoch_acc)
    
    print(f"Época [{epoch+1}/{epochs}] | Loss: {epoch_loss:.4f} | Acurácia Treino: {epoch_acc:.2f}%")


---
### 5. Avaliação da CNN no Conjunto de Teste


In [ ]:
model_cnn.eval()
test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_cnn(images)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

final_cnn_acc = (correct / total) * 100
print(f"📊 Acurácia Final da CNN no Conjunto de Teste: {final_cnn_acc:.2f}%")


---
### 6. Visualização dos Feature Maps Extraídos pelos Filtros Convolucionais
Vamos inspecionar o que a primeira camada de convolução (`conv_block1[0]`) extraiu de uma imagem do número 7!


In [ ]:
# Selecionar uma imagem de teste
sample_img, sample_label = test_dataset[0]  # Imagem individual (1, 28, 28)
input_tensor = sample_img.unsqueeze(0).to(device)  # Adiciona dimensão de batch (1, 1, 28, 28)

# Passar apenas pela primeira camada convolucional
model_cnn.eval()
with torch.no_grad():
    feature_maps = model_cnn.conv_block1[0](input_tensor) # Forma: (1, 16, 28, 28)

feature_maps_np = feature_maps.squeeze(0).cpu().numpy()

# Visualizar os 16 Feature Maps gerados pelos 16 filtros
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle(f"16 Feature Maps Gerados pela 1ª Convolução (Dígito {sample_label})", fontsize=14, fontweight='bold')

for i in range(16):
    row = i // 8
    col = i % 8
    axes[row, col].imshow(feature_maps_np[i], cmap='viridis')
    axes[row, col].set_title(f"Filtro {i+1}", fontsize=9)
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()


---
### 7. Resumo da Comparação: MLP vs CNN no MNIST
- **MLP (Parte 1):** Achata a matriz 2D em 784 entradas, perdendo relações locais.
- **CNN (Parte 2):** Opera sobre matrizes 2D, extrai bordas e texturas com compartilhamento de pesos e gera mapas de características visuais com acurácia superior!
